# 🐾 Train YOLOv10n for PredatorAlert

This notebook trains a YOLOv10n model on the **animals-detection** dataset from Roboflow.

**Animals Detected:** Bear, Elephant, Leopard, Monkey, Tiger (5 classes)

---

## ⚡ Quick Start
1. **Runtime → Change runtime type → GPU (T4)**
2. Run all cells in order
3. Download `best.pt` from the output
4. Copy to your Raspberry Pi's `models/` folder

## Step 1: Check GPU

In [ ]:
!nvidia-smi
print("\n✅ If you see GPU info above, you're good to go!")

## Step 2: Install Dependencies

In [ ]:
# Install Ultralytics (includes YOLOv10 support)
!pip install ultralytics roboflow -q
print("✅ Dependencies installed!")

## Step 3: Download Dataset from Roboflow

⚠️ **Important:** You need a Roboflow API key.
1. Go to https://app.roboflow.com/settings/api
2. Copy your API key
3. Paste it below

In [ ]:
from roboflow import Roboflow

# ⚠️ PASTE YOUR ROBOFLOW API KEY HERE
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # Replace this!

# Download the animals-detection dataset
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("machine-train-ur3hn").project("animals-detection-bsbbi")
version = project.version(3)
dataset = version.download("yolov8")  # YOLOv8 format works with YOLOv10!

print(f"\n✅ Dataset downloaded to: {dataset.location}")

## Step 4: Download YOLOv10n Base Model

In [ ]:
# Download YOLOv10n pretrained weights
!wget -q https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10n.pt
print("✅ YOLOv10n base model downloaded!")

## Step 5: Train the Model 🚀

This will take approximately **30-60 minutes** on a T4 GPU.

In [ ]:
from ultralytics import YOLO

# Load YOLOv10n model
model = YOLO('yolov10n.pt')

# Train on animals-detection dataset
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,           # More epochs = better accuracy
    imgsz=640,            # Image size
    batch=16,             # Batch size (reduce if OOM error)
    patience=15,          # Early stopping
    name='animals_yolov10n',
    device=0,             # GPU
    workers=2,
    plots=True
)

print("\n✅ Training complete!")

## Step 6: Evaluate Model Performance

In [ ]:
# Validate the model
metrics = model.val()

print(f"\n📊 Model Performance:")
print(f"   mAP50: {metrics.box.map50:.2%}")
print(f"   mAP50-95: {metrics.box.map:.2%}")

## Step 7: Test on Sample Images

In [ ]:
import os
from glob import glob
from IPython.display import Image, display

# Get a few test images
test_images = glob(f"{dataset.location}/valid/images/*.jpg")[:3]

# Run inference
for img_path in test_images:
    results = model(img_path)
    
    # Show the image with detections
    for r in results:
        im_array = r.plot()
        # Save and display
        import cv2
        cv2.imwrite('temp_result.jpg', im_array)
        display(Image('temp_result.jpg', width=400))
        print("---")

## Step 8: Download the Trained Model

Download `best.pt` and copy it to your Raspberry Pi's `models/` folder.

In [ ]:
from google.colab import files
import shutil

# Find the best model
best_model_path = 'runs/detect/animals_yolov10n/weights/best.pt'

# Copy to current directory with a clear name
shutil.copy(best_model_path, 'animals_yolov10n_best.pt')

# Download it!
print("📥 Downloading trained model...")
files.download('animals_yolov10n_best.pt')

print("\n✅ Done! Copy this file to your Raspberry Pi:")
print("   Rename it to 'best.pt' and place in 'models/' folder")

## 📋 Next Steps on Raspberry Pi

1. Copy `animals_yolov10n_best.pt` to `raspberry_pi5/models/best.pt`
2. Update `.env` file:
   ```
   MODEL_PATH=models/best.pt
   ```
3. Run your detection script:
   ```bash
   python main.py
   ```